# imz2anndata demo: mouse bladder MSI

This notebook demonstrates how to run `imz2anndata` on the bladder MSI example dataset and how to adjust the main pipeline parameters.

Dataset source:

The mouse bladder MSI dataset is accessible via the ProteomeXchange Consortium under dataset identifier `PXD001283`.

This repository may provide the analysis example code without redistributing the raw MSI data. To run the notebook locally, place the paired dataset files at:

- `data/Bladder-MSI.imzML`
- `data/Bladder-MSI.ibd`


## What the pipeline does

The conversion pipeline:

1. Reads spectra from `imzML` pixel by pixel.
2. Detects whether the input spectra are `centroid` or `profile`.
3. Applies OpenMS peak picking only for `profile` inputs.
4. Aligns spectra into a shared binned `m/z` feature space.
5. Builds an `AnnData` object and writes an `.h5ad` file.


In [ ]:
from pathlib import Path

from imz2anndata import (
    AlignmentConfig,
    PeakPickingConfig,
    PipelineConfig,
    SpatialFilterConfig,
    run_pipeline,
)


In [ ]:
project_root = Path.cwd().resolve().parent if Path.cwd().name in {"examples", "notebooks"} else Path.cwd().resolve()
input_imzml = project_root / "data" / "Bladder-MSI.imzML"
input_ibd = project_root / "data" / "Bladder-MSI.ibd"
output_dir = project_root / "data" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
output_h5ad = output_dir / "bladder_msi_demo_filtered.h5ad"

print("Project root:", project_root)
print("imzML exists:", input_imzml.exists(), input_imzml)
print("ibd exists:", input_ibd.exists(), input_ibd)
print("Output path:", output_h5ad)


## Main parameters

The most important knobs are:

- `dataset_id`: identifier stored in `adata.uns`
- `mz_bin_width`: width of the shared `m/z` bins used during alignment
- `min_feature_occurrence`: minimum number of spectra/pixels in which a feature must appear to be retained
- `enable_peak_picking`: whether peak picking is requested at all
- `peak_picking.snr`: OpenMS signal-to-noise threshold
- `peak_picking.signal_to_noise_window`: OpenMS sliding window size
- `peak_picking.min_intensity`: minimum picked-peak intensity to keep
- `spatial_filter.enabled`: whether to subset the feature space using spatial statistics
- `spatial_filter.assess_only`: whether to compute spatial-filter statistics without subsetting the matrix
- `spatial_filter.min_pixels`: minimum number of positive pixels for a feature
- `spatial_filter.min_total_intensity`: minimum summed intensity for a feature
- `spatial_filter.min_morans_i`: minimum Moran's I score when spatial filtering is enabled

For the bladder MSI demo, the defaults below are intentionally stricter than the package-wide minimal defaults so the resulting feature space is easier to inspect and analyze.


In [ ]:
peak_picking = PeakPickingConfig(
    snr=0.0,
    signal_to_noise_window=100.0,
    min_intensity=0.0,
)

alignment = AlignmentConfig(
    mz_bin_width=0.01,
    min_feature_occurrence=300,
)

spatial_filter = SpatialFilterConfig(
    enabled=True,
    assess_only=False,
    min_pixels=10,
    min_total_intensity=1e4,
    min_morans_i=0.1,
    connectivity=8,
    compute_morans_i=True,
)

config = PipelineConfig(
    input_imzml=input_imzml,
    output_h5ad=output_h5ad,
    dataset_id="bladder_demo",
    peak_picking=peak_picking,
    alignment=alignment,
    spatial_filter=spatial_filter,
    enable_peak_picking=True,
)

config


## Run the conversion

If the input data files are present, this will generate `data/results/bladder_msi_demo_filtered.h5ad`.


In [ ]:
if not input_imzml.exists() or not input_ibd.exists():
    raise FileNotFoundError(
        "Expected local demo files at data/Bladder-MSI.imzML and data/Bladder-MSI.ibd"
    )

adata = run_pipeline(config)
adata


In [ ]:
print("shape:", adata.shape)
print("dataset_id:", adata.uns.get("dataset_id"))
print("spectrum_mode:", adata.uns.get("spectrum_mode"))
print("peak_picking_requested:", adata.uns.get("peak_picking_requested"))
print("peak_picking_enabled:", adata.uns.get("peak_picking_enabled"))
print("peak_picking_strategy:", adata.uns.get("peak_picking_strategy"))
if "selected" in adata.var:
    print("selected feature count:", int(adata.var["selected"].sum()))


In [ ]:
adata.obs.head()


In [ ]:
adata.var.head()


In [ ]:
print("mean raw TIC:", float(adata.obs["tic_raw"].mean()))
print("mean processed TIC:", float(adata.obs["tic_processed"].mean()))
print("mean raw peak count:", float(adata.obs["raw_peak_count"].mean()))
print("mean processed peak count:", float(adata.obs["processed_peak_count"].mean()))


## Equivalent CLI command

The same conversion can be run from the command line:

```bash
imz2anndata data/Bladder-MSI.imzML data/results/bladder_msi_demo_filtered.h5ad \
  --dataset-id bladder_demo \
  --mz-bin-width 0.01 \
  --min-feature-occurrence 300 \
  --peak-picking-snr 0 \
  --signal-to-noise-window 100 \
  --min-intensity 0 \
  --enable-spatial-filtering \
  --min-feature-pixels 10 \
  --min-feature-total-intensity 10000 \
  --min-morans-i 0.1
```
